# Import Ports
This example shows how to import ports. In this example, we are going to

- Download an example board
- Create a configuration file
  - Add a circuit port between two nets
  - Add a circuit port between two pins
  - Add a circuit port between two pin groups
  - Add a circuit port between two coordinates
  - Add a coax port
  - Add a port reference to the nearest pin
  - Add distributed ports
- Import the configuration file

## Perform imports and define constants

Perform required imports.

In [1]:
import json
import toml
from pathlib import Path
import tempfile

from ansys.aedt.core.examples.downloads import download_file
from pyedb import Edb

Define constants.

In [2]:
AEDT_VERSION = "2025.1"
NG_MODE = False

Download the example PCB data.

In [3]:
temp_folder = tempfile.TemporaryDirectory(suffix=".ansys")
file_edb = download_file(source="edb/ANSYS-HSD_V1.aedb", local_path=temp_folder.name)

## Load example layout

In [4]:
edbapp = Edb(file_edb, edbversion=AEDT_VERSION)

PyAEDT INFO: Star initializing Edb 03:52:32.487149


PyAEDT INFO: Logger is initialized in EDB.


PyAEDT INFO: legacy v0.53.0


PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


PyAEDT INFO: Database ANSYS-HSD_V1.aedb Opened in 2025.1


PyAEDT INFO: Cell main Opened


PyAEDT INFO: Builder was initialized.


PyAEDT INFO: EDB initialized.Time lapse 0:00:10.203953


## Create an empty dictionary to host all configurations

In [5]:
cfg = dict()

## Add a circuit port between two nets

Keywords

- **name**. Name of the port.
- **Reference_designator**. Reference designator of the component.
- **type**. Type of the port. Supported types are 'circuit', 'coax'
- **positive_terminal**. Positive terminal of the port. Supported types are 'net', 'pin', 'pin_group', 'coordinates'
- **negative_terminal**. Negative terminal of the port. Supported types are 'net', 'pin', 'pin_group', 'coordinates',
'nearest_pin'

In [6]:
port_1 = {
    "name": "port_1",
    "reference_designator": "X1",
    "type": "circuit",
    "positive_terminal": {"net": "PCIe_Gen4_TX2_N"},
    "negative_terminal": {"net": "GND"},
}

## Add a circuit port between two pins

In [7]:
port_2 = {
    "name": "port_2",
    "reference_designator": "C375",
    "type": "circuit",
    "positive_terminal": {"pin": "1"},
    "negative_terminal": {"pin": "2"},
}

## Add a circuit port between two pin groups

In [8]:
pin_groups = [
    {"name": "U9_5V_1", "reference_designator": "U9", "pins": ["32", "33"]},
    {"name": "U9_GND", "reference_designator": "U9", "net": "GND"},
]

In [9]:
port_3 = {
    "name": "port_3",
    "type": "circuit",
    "positive_terminal": {"pin_group": "U9_5V_1"},
    "negative_terminal": {"pin_group": "U9_GND"},
}

## Add a circuit port between two coordinates

Keywords

- **layer**. Layer on which the terminal is placed
- **point**. XY coordinate the terminal is placed
- **net**. Name of the net the terminal is placed on

In [10]:
port_4 = {
    "name": "port_4",
    "type": "circuit",
    "positive_terminal": {"coordinates": {"layer": "1_Top", "point": ["104mm", "37mm"], "net": "AVCC_1V3"}},
    "negative_terminal": {"coordinates": {"layer": "Inner6(GND2)", "point": ["104mm", "37mm"], "net": "GND"}},
}

## Add a coax port

In [11]:
port_5 = {"name": "port_5", "reference_designator": "U1", "type": "coax", "positive_terminal": {"pin": "AM17"}}

## Add a port reference to the nearest pin

Keywords

- **reference_net**. Name of the reference net
- **search_radius**. Reference pin search radius in meter

In [12]:
port_6 = {
    "name": "port_6",
    "reference_designator": "U15",
    "type": "circuit",
    "positive_terminal": {"pin": "D7"},
    "negative_terminal": {"nearest_pin": {"reference_net": "GND", "search_radius": 5e-3}},
}

## Add distributed ports

Keywords

- **distributed**. Whether to create distributed ports. When set to True, ports are created per pin

In [13]:
ports_distributed = {
    "name": "ports_d",
    "reference_designator": "U7",
    "type": "circuit",
    "distributed": True,
    "positive_terminal": {"net": "VDD_DDR"},
    "negative_terminal": {"net": "GND"},
}

## Add setups in configuration

In [14]:
cfg["pin_groups"] = pin_groups
cfg["ports"] = [port_1, port_2, port_3, port_4, port_5, port_6, ports_distributed]

## Write configuration into as json file

In [15]:
file_json = Path(temp_folder.name) / "edb_configuration.json"
with open(file_json, "w") as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)

Equivalent toml file looks like below 

In [16]:
toml_string = toml.dumps(cfg)
print(toml_string)

[[pin_groups]]
name = "U9_5V_1"
reference_designator = "U9"
pins = [ "32", "33",]

[[pin_groups]]
name = "U9_GND"
reference_designator = "U9"
net = "GND"

[[ports]]
name = "port_1"
reference_designator = "X1"
type = "circuit"

[ports.positive_terminal]
net = "PCIe_Gen4_TX2_N"
[ports.negative_terminal]
net = "GND"
[[ports]]
name = "port_2"
reference_designator = "C375"
type = "circuit"

[ports.positive_terminal]
pin = "1"
[ports.negative_terminal]
pin = "2"
[[ports]]
name = "port_3"
type = "circuit"

[ports.positive_terminal]
pin_group = "U9_5V_1"
[ports.negative_terminal]
pin_group = "U9_GND"
[[ports]]
name = "port_4"
type = "circuit"

[ports.positive_terminal.coordinates]
layer = "1_Top"
point = [ "104mm", "37mm",]
net = "AVCC_1V3"
[ports.negative_terminal.coordinates]
layer = "Inner6(GND2)"
point = [ "104mm", "37mm",]
net = "GND"
[[ports]]
name = "port_5"
reference_designator = "U1"
type = "coax"

[ports.positive_terminal]
pin = "AM17"
[[ports]]
name = "port_6"
reference_designator =

## Import configuration into example layout

In [17]:
edbapp.configuration.load(config_file=file_json)
edbapp.configuration.run()

PyAEDT INFO: Updating boundaries finished. Time lapse 0:00:00.015640


PyAEDT INFO: Updating nets finished. Time lapse 0:00:00


PyAEDT INFO: Updating components finished. Time lapse 0:00:00


PyAEDT INFO: Creating pin groups finished. Time lapse 0:00:00.046861


PyAEDT INFO: Placing sources finished. Time lapse 0:00:00


...

PyAEDT INFO: Applying modeler finished. Time lapse 0:00:00.562538


PyAEDT INFO: Placing ports finished. Time lapse 0:00:00.906313


PyAEDT INFO: Placing probes finished. Time lapse 0:00:00


PyAEDT INFO: Applying operations finished. Time lapse 0:00:00


True

## Review

In [18]:
edbapp.ports

{'port_1': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x237575c0a60>,
 'port_2': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x2374ea25c90>,
 'port_3': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x1f6944fc940>,
 'port_4': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x1f6944d2500>,
 'port_5': <pyedb.dotnet.database.edb_data.ports.CoaxPort at 0x1f6944fd960>,
 'port_6': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x1f6944fc850>,
 'U7_VDD_DDR_T9': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x237575c1c60>,
 'U7_VDD_DDR_R1': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x237575c21a0>,
 'U7_VDD_DDR_L9': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x237575c1060>,
 'U7_VDD_DDR_L1': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x237575c31c0>,
 'U7_VDD_DDR_J9': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x237575c1ae0>,
 'U7_VDD_DDR_J8': <pyedb.dotnet.database.edb_data.ports.CircuitPort at 0x237575c1240>,
 

## Save and close Edb
The temporary folder will be deleted once the execution of this script is finished. Replace **edbapp.save()** with
**edbapp.save_as("C:/example.aedb")** to keep the example project.

In [19]:
edbapp.save()
edbapp.close()

PyAEDT INFO: EDB file save time: 0.00ms


PyAEDT INFO: EDB file release time: 0.00ms


True